# STRING Validation — Leakage-Fixed Model

In [ ]:
!pip install requests pandas numpy matplotlib -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import json

from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/'

In [ ]:
# Load data
with pd.HDFStore(DRIVE_PATH + 'gene_network_data.h5') as store:
    tec = store['TEC']
np_tec_abs = np.abs(tec.to_numpy(copy=True))
gene_names = tec.index.tolist()
n_genes = len(gene_names)
del tec
print(f'Loaded {n_genes} genes')

In [ ]:
# Load leakage-fixed embeddings
embeddings_np = np.load(DRIVE_PATH + 'noleak_gnn_embeddings.npy')
print(f'Embeddings: {embeddings_np.shape}')
print(f'Embedding std: {embeddings_np.std():.4f}')

# Training adjacency to exclude existing edges from predictions
threshold = 0.75
adj_matrix = (np_tec_abs > threshold).astype(float)
np.fill_diagonal(adj_matrix, 0)

In [ ]:
# Top 5000 novel predictions
similarity_matrix = cosine_similarity(embeddings_np)
np.fill_diagonal(similarity_matrix, 0)
similarity_matrix[adj_matrix > 0] = 0  # exclude training edges

high_sim_idx = np.where(similarity_matrix > 0.95)
predictions = []
for i, j in zip(high_sim_idx[0], high_sim_idx[1]):
    if i < j:
        predictions.append((gene_names[i], gene_names[j], similarity_matrix[i, j]))

predictions.sort(key=lambda x: x[2], reverse=True)
top_predictions = predictions[:5000]
print(f'Novel predictions (>0.95): {len(predictions)}')
print(f'Using top {len(top_predictions)} for validation')

In [ ]:
# Download STRING protein info
STRING_INFO_URL  = 'https://stringdb-downloads.org/download/protein.info.v12.0/9606.protein.info.v12.0.txt.gz'
STRING_LINKS_URL = 'https://stringdb-downloads.org/download/protein.links.v12.0/9606.protein.links.v12.0.txt.gz'

print('Downloading STRING protein info...')
info_df = pd.read_csv(STRING_INFO_URL, sep='\t', compression='gzip')
gene_to_string = dict(zip(info_df['preferred_name'], info_df['#string_protein_id']))
our_genes_in_string = [g for g in gene_names if g in gene_to_string]
print(f'Our genes in STRING: {len(our_genes_in_string)}/{n_genes}')

print('Downloading STRING links...')
links_df = pd.read_csv(STRING_LINKS_URL, sep=' ', compression='gzip')
high_conf = links_df[links_df['combined_score'] >= 700]
string_pairs = set()
for _, row in high_conf.iterrows():
    string_pairs.add((min(row['protein1'], row['protein2']),
                      max(row['protein1'], row['protein2'])))
print(f'STRING high-confidence pairs: {len(string_pairs)}')

In [ ]:
# Validate predictions
validated, not_in_string, total_checkable = 0, 0, 0
validated_pairs, not_validated_pairs = [], []

for gene1, gene2, score in top_predictions:
    id1 = gene_to_string.get(gene1)
    id2 = gene_to_string.get(gene2)
    if id1 is None or id2 is None:
        not_in_string += 1
        continue
    total_checkable += 1
    pair = (min(id1, id2), max(id1, id2))
    if pair in string_pairs:
        validated += 1
        validated_pairs.append((gene1, gene2, score))
    else:
        not_validated_pairs.append((gene1, gene2, score))

precision = validated / total_checkable if total_checkable > 0 else 0
print(f'\n=== STRING VALIDATION RESULTS ===')
print(f'Checkable pairs: {total_checkable}')
print(f'Validated: {validated}')
print(f'Precision: {precision*100:.2f}%')
print(f'\nSample validated pairs:')
for g1, g2, s in validated_pairs[:10]:
    print(f'  {g1} -- {g2}: {s:.4f}')

In [ ]:
# Random baseline
np.random.seed(42)
rand_validated, rand_checkable = 0, 0
string_genes = list(our_genes_in_string)
for _ in range(min(5000, total_checkable)):
    g1, g2 = np.random.choice(string_genes, 2, replace=False)
    id1, id2 = gene_to_string.get(g1), gene_to_string.get(g2)
    if id1 and id2:
        rand_checkable += 1
        if (min(id1, id2), max(id1, id2)) in string_pairs:
            rand_validated += 1

random_precision = rand_validated / rand_checkable if rand_checkable > 0 else 0
enrichment = precision / random_precision if random_precision > 0 else float('inf')

print(f'GNN Precision:    {precision*100:.2f}%')
print(f'Random Precision: {random_precision*100:.2f}%')
print(f'Enrichment:       {enrichment:.1f}x')

In [ ]:
# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(['GNN Predictions', 'Random Baseline'],
            [precision * 100, random_precision * 100],
            color=['#2196F3', '#FF5722'], width=0.5)
axes[0].set_ylabel('Precision (% validated by STRING)', fontsize=12)
axes[0].set_title('GNN Prediction Precision vs Random Baseline', fontsize=13)
for i, v in enumerate([precision * 100, random_precision * 100]):
    axes[0].text(i, v + 0.1, f'{v:.2f}%', ha='center', fontsize=12, fontweight='bold')

val_scores     = [s for _, _, s in validated_pairs]
not_val_scores = [s for _, _, s in not_validated_pairs[:len(validated_pairs)]]
axes[1].hist(val_scores,     bins=30, alpha=0.6, color='#2196F3', label=f'STRING validated (n={len(val_scores)})')
axes[1].hist(not_val_scores, bins=30, alpha=0.6, color='#FF5722', label=f'Not in STRING (n={len(not_val_scores)})')
axes[1].set_xlabel('GNN Similarity Score', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Similarity Score Distribution', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.savefig('noleak_string_validation_results.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: noleak_string_validation_results.png')

In [ ]:
# Save results
results = {
    'leakage_fixed': True,
    'total_predictions_checked': len(top_predictions),
    'checkable_pairs': total_checkable,
    'string_validated': validated,
    'gnn_precision': precision,
    'random_precision': random_precision,
    'enrichment_over_random': enrichment,
    'top_validated_pairs': [(g1, g2, float(s)) for g1, g2, s in validated_pairs[:20]]
}
with open('noleak_string_validation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved: noleak_string_validation_results.json')